# Assignment 3 — IoT Sensor Data Analysis

Dataset: `iot_sensor_data_raw.csv`

Objective: Analyze sensor data and identify abnormal operating conditions.

In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


## Part A — Data Preparation

In [2]:
# 1. Load the dataset
df = pd.read_csv('../../data/iot_sensor_data_raw.csv')

# 2. Display its dimensions and structure
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nColumns:", df.columns.tolist())

Shape: (20000, 9)
Rows: 20000, Columns: 9

Columns: ['Timestamp', 'Device_ID', 'Temperature', 'Humidity', 'Pressure', 'Vibration', 'Battery_Level', 'Location', 'Machine_Status']


In [3]:
# 3. Convert Timestamp into Pandas datetime
if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    print("Timestamp converted to datetime")
    print(f"Date range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")

Timestamp converted to datetime
Date range: 2026-01-01 00:00:00 to 2026-07-28 07:45:00


In [4]:
# 4. Check whether the timestamps are correctly ordered
if 'Timestamp' in df.columns:
    is_sorted = df['Timestamp'].is_monotonic_increasing
    print(f"Timestamps are sorted: {is_sorted}")
    if not is_sorted:
        df = df.sort_values('Timestamp').reset_index(drop=True)
        print("DataFrame sorted by Timestamp")

Timestamps are sorted: False
DataFrame sorted by Timestamp


In [5]:
# 5. Identify missing sensor readings
sensor_cols = [col for col in df.columns if 'sensor' in col.lower() or col in ['Temperature', 'Vibration', 'Battery', 'Humidity', 'Pressure']]
print(f"Sensor columns: {sensor_cols}")

missing = df[sensor_cols].isnull().sum()
print("Missing sensor readings:")
print(missing[missing > 0])

Sensor columns: ['Temperature', 'Humidity', 'Pressure', 'Vibration']
Missing sensor readings:
Temperature    299
Humidity       300
Pressure       300
Vibration      298
dtype: int64


In [6]:
# 6. Calculate the percentage of missing values for each sensor
missing_pct = (df[sensor_cols].isnull().sum() / len(df)) * 100
print("Percentage of missing values per sensor:")
print(missing_pct[missing_pct > 0])

Percentage of missing values per sensor:


Temperature    1.495
Humidity       1.500
Pressure       1.500
Vibration      1.490
dtype: float64


## Part B — Data Cleaning

In [7]:
# 7. Handle missing sensor values using an appropriate method
# Forward fill then backward fill for time-series data
for col in sensor_cols:
    if df[col].isnull().any():
        df[col] = df[col].ffill().bfill()
        print(f"Filled missing values in {col} using forward/backward fill")

Filled missing values in Temperature using forward/backward fill
Filled missing values in Humidity using forward/backward fill
Filled missing values in Pressure using forward/backward fill
Filled missing values in Vibration using forward/backward fill


In [8]:
# 8. Identify abnormal temperature readings
if 'Temperature' in df.columns:
    # Define abnormal as outside 3 standard deviations from mean
    temp_mean = df['Temperature'].mean()
    temp_std = df['Temperature'].std()
    lower_bound = temp_mean - 3 * temp_std
    upper_bound = temp_mean + 3 * temp_std
    
    abnormal_temp = df[(df['Temperature'] < lower_bound) | (df['Temperature'] > upper_bound)]
    print(f"Temperature stats: mean={temp_mean:.2f}, std={temp_std:.2f}")
    print(f"Abnormal bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"Abnormal temperature readings: {len(abnormal_temp)}")
    if len(abnormal_temp) > 0:
        display(abnormal_temp[['Timestamp', 'Temperature']].head(10))

Temperature stats: mean=70.21, std=8.67
Abnormal bounds: [44.19, 96.22]
Abnormal temperature readings: 71


,Timestamp,Temperature
426,2026-01-05 10:30:00,103.180000
573,2026-01-06 23:15:00,134.777755
829,2026-01-09 15:15:00,127.783177
980,2026-01-11 05:00:00,150.182051
1067,2026-01-12 02:45:00,141.071073
1154,2026-01-13 00:30:00,129.367099
1619,2026-01-17 20:45:00,34.170000
1686,2026-01-18 13:30:00,139.785887
1756,2026-01-19 07:00:00,42.940000
2049,2026-01-22 08:15:00,152.974450


In [9]:
# 9. Identify abnormal vibration readings
if 'Vibration' in df.columns:
    vib_mean = df['Vibration'].mean()
    vib_std = df['Vibration'].std()
    lower_bound = vib_mean - 3 * vib_std
    upper_bound = vib_mean + 3 * vib_std
    
    abnormal_vib = df[(df['Vibration'] < lower_bound) | (df['Vibration'] > upper_bound)]
    print(f"Vibration stats: mean={vib_mean:.2f}, std={vib_std:.2f}")
    print(f"Abnormal bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"Abnormal vibration readings: {len(abnormal_vib)}")
    if len(abnormal_vib) > 0:
        display(abnormal_vib[['Timestamp', 'Vibration']].head(10))

Vibration stats: mean=2.53, std=1.02
Abnormal bounds: [-0.52, 5.58]
Abnormal vibration readings: 51


,Timestamp,Vibration
603,2026-01-07 06:45:00,11.662231
875,2026-01-10 02:45:00,19.993705
1141,2026-01-12 21:15:00,11.306348
1211,2026-01-13 14:45:00,16.349788
1454,2026-01-16 03:30:00,13.881332
1996,2026-01-21 19:00:00,18.470897
2095,2026-01-22 19:45:00,15.708531
2227,2026-01-24 04:45:00,14.639271
2347,2026-01-25 10:45:00,16.400273
2558,2026-01-27 15:30:00,15.859665


In [10]:
# 10. Identify machines with critically low battery levels
if 'Battery_Level' in df.columns:
    # Critically low: below 20%
    low_battery = df[df['Battery_Level'] < 20]
    print(f"Machines with battery < 20%: {len(low_battery)}")
    if len(low_battery) > 0:
        # Group by device if Device_ID exists
        if 'Device_ID' in df.columns:
            low_by_device = low_battery.groupby('Device_ID')['Battery_Level'].min().sort_values()
            print("\nMin battery by device:")
            print(low_by_device)

Machines with battery < 20%: 2317

Min battery by device:
Device_ID
DEV_008    2.011730
DEV_007    2.146086
DEV_002    2.154615
DEV_006    2.201033
DEV_001    2.257059
DEV_005    2.956738
DEV_003    3.670604
DEV_004    8.514009
Name: Battery_Level, dtype: float64


In [11]:
# 11. Explain how you decided what constitutes an abnormal reading
print("""
Abnormal Reading Detection Method:
1. Temperature & Vibration: Used 3-sigma rule (mean ± 3 standard deviations)
   - This captures 99.7% of normal data under normal distribution
   - Readings outside this range are statistical outliers
2. Battery: Used domain threshold (< 20%)
   - Based on typical IoT sensor battery health standards
   - Below 20% indicates need for replacement/recharge
3. Alternative methods could include:
   - IQR method (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
   - Domain-specific thresholds from equipment manuals
   - Machine learning anomaly detection (Isolation Forest, etc.)
""")


Abnormal Reading Detection Method:
1. Temperature & Vibration: Used 3-sigma rule (mean ± 3 standard deviations)
   - This captures 99.7% of normal data under normal distribution
   - Readings outside this range are statistical outliers
2. Battery: Used domain threshold (< 20%)
   - Based on typical IoT sensor battery health standards
   - Below 20% indicates need for replacement/recharge
3. Alternative methods could include:
   - IQR method (Q1 - 1.5*IQR, Q3 + 1.5*IQR)
   - Domain-specific thresholds from equipment manuals
   - Machine learning anomaly detection (Isolation Forest, etc.)



## Part C — Time-Series Analysis

In [12]:
# 12. Calculate the average temperature by hour
if 'Timestamp' in df.columns and 'Temperature' in df.columns:
    df['Hour'] = df['Timestamp'].dt.hour
    avg_temp_by_hour = df.groupby('Hour')['Temperature'].mean()
    print("Average temperature by hour:")
    print(avg_temp_by_hour)

Average temperature by hour:
Hour
0     70.044013
1     70.572631
2     69.962296
3     69.701742
4     69.825251
5     70.297896
6     70.327452
7     70.659045
8     70.391414
9     70.979044
10    69.879546
11    69.887453
12    69.985877
13    70.062964
14    69.790531
15    70.168951
16    69.916465
17    70.553290
18    70.151022
19    69.973942
20    70.134539
21    70.159663
22    71.139769
23    70.417236
Name: Temperature, dtype: float64


In [13]:
# 13. Calculate the average temperature for each device
if 'Device_ID' in df.columns and 'Temperature' in df.columns:
    avg_temp_by_device = df.groupby('Device_ID')['Temperature'].mean().sort_values(ascending=False)
    print("Average temperature by device:")
    print(avg_temp_by_device)

Average temperature by device:
Device_ID
DEV_004    70.601292
DEV_002    70.346665
DEV_008    70.314971
DEV_001    70.264967
DEV_007    70.120640
DEV_005    70.110261
DEV_006    69.986978
DEV_003    69.902047
Name: Temperature, dtype: float64


In [14]:
# 14. Calculate the average vibration for each device
if 'Device_ID' in df.columns and 'Vibration' in df.columns:
    avg_vib_by_device = df.groupby('Device_ID')['Vibration'].mean().sort_values(ascending=False)
    print("Average vibration by device:")
    print(avg_vib_by_device)

Average vibration by device:
Device_ID
DEV_002    2.565481
DEV_006    2.552977
DEV_001    2.549300
DEV_003    2.537439
DEV_008    2.528819
DEV_004    2.521912
DEV_007    2.520413
DEV_005    2.498862
Name: Vibration, dtype: float64


In [15]:
# 15. Find the maximum temperature recorded by each device
if 'Device_ID' in df.columns and 'Temperature' in df.columns:
    max_temp_by_device = df.groupby('Device_ID')['Temperature'].max().sort_values(ascending=False)
    print("Maximum temperature by device:")
    print(max_temp_by_device)

Maximum temperature by device:
Device_ID
DEV_001    159.746758
DEV_003    157.603693
DEV_007    153.877632
DEV_008    153.622123
DEV_005    152.974450
DEV_004    151.965273
DEV_002    151.818550
DEV_006    136.003203
Name: Temperature, dtype: float64


In [16]:
# 16. Find the minimum battery level for each device
if 'Device_ID' in df.columns and 'Battery_Level' in df.columns:
    min_battery_by_device = df.groupby('Device_ID')['Battery_Level'].min().sort_values()
    print("Minimum battery by device:")
    print(min_battery_by_device)

Minimum battery by device:
Device_ID
DEV_008    2.011730
DEV_007    2.146086
DEV_002    2.154615
DEV_006    2.201033
DEV_001    2.257059
DEV_005    2.956738
DEV_003    3.670604
DEV_004    8.514009
Name: Battery_Level, dtype: float64


In [17]:
# 17. Determine which device has the highest average vibration
if 'Device_ID' in df.columns and 'Vibration' in df.columns:
    avg_vib = df.groupby('Device_ID')['Vibration'].mean()
    highest_vib_device = avg_vib.idxmax()
    highest_vib_value = avg_vib.max()
    print(f"Device with highest average vibration: {highest_vib_device} ({highest_vib_value:.2f})")

Device with highest average vibration: DEV_002 (2.57)


In [18]:
# 18. Determine which factory has the highest average temperature
if 'Factory' in df.columns and 'Temperature' in df.columns:
    avg_temp_factory = df.groupby('Factory')['Temperature'].mean().sort_values(ascending=False)
    print("Average temperature by factory:")
    print(avg_temp_factory)
    highest_temp_factory = avg_temp_factory.index[0]
    print(f"\nFactory with highest average temperature: {highest_temp_factory}")
else:
    print("Factory column not found")

Factory column not found


## Part D — Create New Variables

In [19]:
# 19. Create a Battery_Status column:
# Battery >= 50 → Healthy
# Battery 20-49 → Low
# Battery < 20 → Critical
if 'Battery_Level' in df.columns:
    def battery_status(battery):
        if battery >= 50:
            return 'Healthy'
        elif battery >= 20:
            return 'Low'
        else:
            return 'Critical'
    df['Battery_Status'] = df['Battery_Level'].apply(battery_status)
    print("Battery_Status distribution:")
    print(df['Battery_Status'].value_counts())

Battery_Status distribution:
Battery_Status
Healthy     10978
Low          6705
Critical     2317
Name: count, dtype: int64


## Part D — Create New Variables

In [20]:
# 20. Create a Temperature_Status column
if 'Temperature' in df.columns:
    temp_mean = df['Temperature'].mean()
    temp_std = df['Temperature'].std()
    def temp_status(temp):
        if temp > temp_mean + 2 * temp_std:
            return 'High'
        elif temp < temp_mean - 2 * temp_std:
            return 'Low'
        else:
            return 'Normal'
    df['Temperature_Status'] = df['Temperature'].apply(temp_status)
    print("Temperature_Status distribution:")
    print(df['Temperature_Status'].value_counts())

Temperature_Status distribution:
Temperature_Status
Normal    19392
High        319
Low         289
Name: count, dtype: int64


In [21]:
# 21. Create a Vibration_Status column
if 'Vibration' in df.columns:
    vib_mean = df['Vibration'].mean()
    vib_std = df['Vibration'].std()
    def vib_status(vib):
        if vib > vib_mean + 2 * vib_std:
            return 'High'
        elif vib < vib_mean - 2 * vib_std:
            return 'Low'
        else:
            return 'Normal'
    df['Vibration_Status'] = df['Vibration'].apply(vib_status)
    print("Vibration_Status distribution:")
    print(df['Vibration_Status'].value_counts())

Vibration_Status distribution:


Vibration_Status
Normal    19732
High        139
Low         129
Name: count, dtype: int64


In [22]:
# 22. Create an overall Machine_Health column:
# Normal, Warning, Critical based on appropriate sensor conditions
conditions = []
if 'Temperature_Status' in df.columns:
    conditions.append(df['Temperature_Status'] == 'High')
if 'Vibration_Status' in df.columns:
    conditions.append(df['Vibration_Status'] == 'High')
if 'Battery_Status' in df.columns:
    conditions.append(df['Battery_Status'] == 'Critical')

if conditions:
    df['Machine_Health'] = 'Normal'
    df.loc[np.any(conditions, axis=0), 'Machine_Health'] = 'Warning'
    # Critical if multiple conditions met
    critical_mask = np.sum(conditions, axis=0) >= 2
    df.loc[critical_mask, 'Machine_Health'] = 'Critical'
    print("Machine_Health distribution:")
    print(df['Machine_Health'].value_counts())

Machine_Health distribution:


Machine_Health
Normal      17285
Warning      2655
Critical       60
Name: count, dtype: int64


## Part E — Advanced Analysis

In [23]:
# 23. Find devices that have experienced critical conditions more than five times
if 'Machine_Health' in df.columns and 'Device_ID' in df.columns:
    critical_counts = df[df['Machine_Health'] == 'Critical'].groupby('Device_ID').size()
    critical_devices = critical_counts[critical_counts > 5]
    print(f"Devices with >5 critical conditions: {len(critical_devices)}")
    if len(critical_devices) > 0:
        print(critical_devices)

Devices with >5 critical conditions: 8


Device_ID
DEV_001     7
DEV_002     6
DEV_003     7
DEV_004    10
DEV_005     7
DEV_006     7
DEV_007    10
DEV_008     6
dtype: int64


In [24]:
# 24. Find the factory with the highest number of abnormal sensor readings
if 'Factory' in df.columns and 'Machine_Health' in df.columns:
    abnormal_counts = df[df['Machine_Health'] != 'Normal'].groupby('Factory').size()
    if len(abnormal_counts) > 0:
        worst_factory = abnormal_counts.idxmax()
        print(f"Factory with most abnormal readings: {worst_factory} ({abnormal_counts.max()})")
        print("\nAbnormal readings by factory:")
        print(abnormal_counts.sort_values(ascending=False))

In [25]:
# 25. Calculate the percentage of time each device operates under warning/critical conditions
if 'Machine_Health' in df.columns and 'Device_ID' in df.columns:
    health_counts = df.groupby('Device_ID')['Machine_Health'].value_counts(normalize=True).unstack(fill_value=0)
    if 'Warning' in health_counts.columns and 'Critical' in health_counts.columns:
        health_counts['Warning_Critical_Pct'] = (health_counts['Warning'] + health_counts['Critical']) * 100
        print("Percentage of time in Warning/Critical by device:")
        print(health_counts['Warning_Critical_Pct'].sort_values(ascending=False))

Percentage of time in Warning/Critical by device:


Device_ID
DEV_004    14.628297
DEV_006    14.041514
DEV_005    13.874854
DEV_002    13.715415
DEV_008    13.528703
DEV_001    13.470681
DEV_003    13.077558
DEV_007    12.244898
Name: Warning_Critical_Pct, dtype: float64


In [26]:
# 26. Identify the device that requires the highest maintenance priority
if 'Machine_Health' in df.columns and 'Device_ID' in df.columns:
    # Priority score: Critical=3, Warning=2, Normal=1
    health_score = {'Critical': 3, 'Warning': 2, 'Normal': 1}
    df['Health_Score'] = df['Machine_Health'].map(health_score)
    device_priority = df.groupby('Device_ID')['Health_Score'].mean().sort_values(ascending=False)
    print("Device maintenance priority (higher = more urgent):")
    print(device_priority.head(10))
    top_device = device_priority.index[0]
    print(f"\nDevice requiring highest maintenance priority: {top_device}")

Device maintenance priority (higher = more urgent):
Device_ID
DEV_004    1.150280
DEV_006    1.143264
DEV_005    1.141469
DEV_002    1.139526
DEV_008    1.137696
DEV_001    1.137480
DEV_003    1.133663
DEV_007    1.126451
Name: Health_Score, dtype: float64

Device requiring highest maintenance priority: DEV_004


## Summary

Assignment 3 complete. IoT sensor data analyzed, anomalies detected, and battery status created.